# 5.1 — Background contribution varies by perturbation modality

Compute and visualise $\Delta_{\mathrm{bg}}(\mathrm{mAP}) = \mathrm{mAP}(C) - \mathrm{mAP}(S)$ for every (encoder, dataset) configuration.

**Paired views (per cell):**
- **C** — *Crop*: full image patch around the cell (cell + surround).
- **S** — *Segmented*: same crop with everything outside the cell's segmentation mask replaced with black.
- **CD** — *Crop + corner Density patches*: C with synthetic density-like texture added in the corners (probe: does extra surround signal raise performance?).
- **SD** — *Segmented + corner Density patches*: S with the same corner patches (probe: does the encoder pick up density signal even when the cell is the only real content?).

Because C and S contain the same central cell and differ only in whether surrounding pixels are retained or masked, $\Delta_{\mathrm{bg}} = \mathrm{mAP}(C) - \mathrm{mAP}(S)$ isolates the performance attributable to non-cell content. CD and SD are auxiliary probes for the appendix table.

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- Global config (put in a shared module or top of notebook) ---
TEXTWIDTH_PT = 397.48499  # from \the\textwidth
TEXTWIDTH_IN = TEXTWIDTH_PT / 72.27
FULLWIDTH_IN = TEXTWIDTH_IN
HALFWIDTH_IN = TEXTWIDTH_IN / 2

def neurips_style():
    mpl.rcParams.update({
        # Font: match LaTeX document
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "mathtext.fontset": "dejavusans",

        # Font sizes: targeted so they're readable at final print size
        "font.size": 7,
        "axes.titlesize": 7,
        "axes.labelsize": 7,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,

        # Clean aesthetics
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.linewidth": 0.6,
        "xtick.major.width": 0.6,
        "ytick.major.width": 0.6,
        "lines.linewidth": 1.0,
        "patch.linewidth": 0.6,

        # Export
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.01,
        "pdf.fonttype": 42,   # TrueType — keeps fonts editable
        "ps.fonttype": 42,
    })

neurips_style()

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch

DATA_DIR = Path('../03_phenotypic_activity')
# CPM eval run through the canonical DL pipeline (Spherize on controls + PCA + Harmony),
# matching the substantive normalization of every encoder. Replaces the old per-plate
# MAD-vs-controls path which gave JUMP a structural advantage due to <5 ctrl/plate fallback.
CPM_CSV = Path('../baselines/phenotypic_activity_cp_measure_dl_pipeline.csv')
FIG_DIR = Path('.')
FIG_DIR.mkdir(exist_ok=True)

DATASETS = ['RxRx1', 'Rxrx3C', 'JUMP']  # ordered by hypothesised Δ_bg magnitude
ENCODERS = ['SubCell', 'DINO', 'OpenPhenom']  # canonical encoder order across all CP-BG-Bench plots

ENCODER_DISPLAY = {
    'SubCell':    'SubCell',
    'DINO':       'DINO\nViT-B/16',
    'OpenPhenom': 'Open\nPhenom',
    'CPM':        'CPM',
}

DATASET_CSV = {
    'RxRx1': DATA_DIR / 'phenotypic_activity_rxrx1.csv',
    'Rxrx3C': DATA_DIR / 'phenotypic_activity_rxrx3c.csv',
    'JUMP': DATA_DIR / 'phenotypic_activity_jump.csv',
}

DATASET_LABEL = {
    'RxRx1': 'RxRx1\n(siRNA, HEPG2)',
    'Rxrx3C': 'RxRx3-core\n(CRISPR-KO, HUVEC)',
    'JUMP': 'JUMP-CP\n(compounds, U2OS)',
}

In [ ]:
VIEWS = ['C', 'CD', 'S', 'SD']

frames = []
for ds, path in DATASET_CSV.items():
    df = pd.read_csv(path)
    df['dataset'] = ds
    df['encoder'] = df['group'].str.split(' — ').str[1]
    frames.append(df)

raw = pd.concat(frames, ignore_index=True)
raw = raw[raw['embedding'].isin(['pre_harmony', 'post_harmony'])]
raw = raw[raw['view'].isin(VIEWS)]
raw = raw[raw['encoder'].isin(ENCODERS)].copy()
raw.head()

In [ ]:
# Pivot to (dataset, encoder, embedding) × view → wide table with all 4 views
wide = raw.pivot_table(
    index=['dataset', 'encoder', 'embedding'],
    columns='view',
    values='mean_mAP',
).reset_index()

# Primary quantity: Δ_bg = mAP(C) − mAP(S). Auxiliary: density-patch deltas.
wide['delta_bg'] = wide['C'] - wide['S']
wide['delta_CD_C'] = wide['CD'] - wide['C']  # density signal added on top of full crop
wide['delta_SD_S'] = wide['SD'] - wide['S']  # density signal added on top of cell-only

dataset_order = pd.Categorical(wide['dataset'], categories=DATASETS, ordered=True)
encoder_order = pd.Categorical(wide['encoder'], categories=ENCODERS, ordered=True)
wide = wide.assign(dataset=dataset_order, encoder=encoder_order)
wide = wide.sort_values(['embedding', 'dataset', 'encoder']).reset_index(drop=True)
wide

In [ ]:
# Dataset-mean Δ_bg per Harmony stage
summary = (
    wide.groupby(['embedding', 'dataset'], observed=True)['delta_bg']
    .agg(['mean', 'min', 'max', 'std'])
    .round(4)
)
print('Δ_bg(mAP) summary by dataset and Harmony stage:')
print(summary)

# Per-config Δ_bg, post-Harmony (primary framing for §5.1)
post = wide[wide['embedding'] == 'post_harmony'].copy()
print('\nPer-config Δ_bg (post-Harmony):')
print(post[['dataset', 'encoder', 'C', 'S', 'delta_bg']].round(4).to_string(index=False))

In [ ]:
# Non-overlap diagnostics (post-Harmony, primary framing)
post_grouped = post.groupby('dataset', observed=True)['delta_bg']
mins = post_grouped.min()
maxs = post_grouped.max()

print('Post-Harmony Δ_bg ranges per dataset:')
for ds in DATASETS:
    print(f'  {ds:7s}  min={mins[ds]:+.4f}  max={maxs[ds]:+.4f}')

print()
print(f"RxRx1 vs JUMP   non-overlap: {mins['RxRx1']  > maxs['JUMP']}    (margin = {mins['RxRx1']  - maxs['JUMP']:+.4f})")
print(f"RxRx1 vs Rxrx3C non-overlap: {mins['RxRx1']  > maxs['Rxrx3C']}  (margin = {mins['RxRx1']  - maxs['Rxrx3C']:+.4f})")
print(f"Rxrx3C vs JUMP  non-overlap: {mins['Rxrx3C'] > maxs['JUMP']}    (margin = {mins['Rxrx3C'] - maxs['JUMP']:+.4f})")

In [ ]:
VIEW_COLOR = {
    'C':  '#2271b2',
    'CD': '#7fceaf',
    'S':  '#e84b4b',
    'SD': '#ff913a',
}
VIEWS_ORDERED = ['C', 'CD', 'S', 'SD']
CPM_COLOR = '#888888'

# Load real cp_measure baseline values from copairs eval CSV.
# CSV's group prefix uses 'Rxrx1' (single capital R) — normalise to DATASETS keys.
_CPM_DATASET_NORM = {'Rxrx1': 'RxRx1', 'Rxrx3C': 'Rxrx3C', 'JUMP': 'JUMP'}
_cpm_df = pd.read_csv(CPM_CSV)
_cpm_df['dataset'] = _cpm_df['group'].str.split(' — ').str[0].map(_CPM_DATASET_NORM)
CPM_VALUES = {
    (row['dataset'], row['embedding']): float(row['mean_mAP'])
    for _, row in _cpm_df.iterrows()
}

bar_w = 1.0
group_inner_w = len(VIEWS_ORDERED) * bar_w
group_gap = 1.5            # spacing between groups (encoders and CPM)
group_step = group_inner_w + group_gap

# 3 encoder groups (4 bars each) + 1 CPM group (1 single bar). CPM uses the
# same gap as between encoder groups so the layout is uniform.
encoder_centers = np.array([
    ei * group_step + (group_inner_w - bar_w) / 2
    for ei in range(len(ENCODERS))
])
cpm_x = len(ENCODERS) * group_step
group_centers_all = np.concatenate([encoder_centers, [cpm_x]])
group_labels_all = list(ENCODERS) + ['CPM']

bar_x = {
    (enc, view): ei * group_step + vi * bar_w
    for ei, enc in enumerate(ENCODERS)
    for vi, view in enumerate(VIEWS_ORDERED)
}


def label_panel(ax, label, x=-0.15, y=1.2):
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=10, fontweight="bold", va="top", ha="left")


def render_modality_figure(stage: str, out_stem: str, ymax: float | None = None):
    """Render the 3-panel paired-views figure for one Harmony stage."""
    df = wide[wide['embedding'] == stage]
    if ymax is None:
        ymax = df[VIEWS_ORDERED].values.max() * 1.05

    fig, axes = plt.subplots(1, 3, figsize=(FULLWIDTH_IN, 2))
    for ax, ds in zip(axes, DATASETS):
        sub = df[df['dataset'] == ds].set_index('encoder').loc[ENCODERS]

        for enc in ENCODERS:
            for view in VIEWS_ORDERED:
                ax.bar(
                    bar_x[(enc, view)],
                    sub.loc[enc, view],
                    width=bar_w,
                    color=VIEW_COLOR[view],
                    edgecolor='black',
                    linewidth=0.4,
                )

        cpm_val = CPM_VALUES[(ds, stage)]
        ax.bar(
            cpm_x, cpm_val, width=bar_w,
            color=CPM_COLOR, edgecolor='black', linewidth=0.4,
        )
        ax.axhline(y=cpm_val, color='gray', linewidth=0.5, linestyle='--', zorder=0)

        all_x = [bar_x[(enc, v)] for enc in ENCODERS for v in VIEWS_ORDERED] + [cpm_x]
        ax.set_xticks(all_x)
        ax.set_xticklabels([''] * len(all_x))

        for centre, enc in zip(group_centers_all, group_labels_all):
            ax.text(
                centre, -0.10, ENCODER_DISPLAY[enc],
                transform=ax.get_xaxis_transform(),
                ha='center', va='top',
            )

        ax.set_title(DATASET_LABEL[ds])
        ax.axhline(0, color='gray', linewidth=0.5)
        ax.set_xlim(-bar_w, cpm_x + bar_w)
        ax.set_ylim(0, ymax)

    for ax, lbl in zip(axes, "ABC"):
        label_panel(ax, lbl)

    legend_handles = [Patch(color=VIEW_COLOR[v], label=v) for v in VIEWS_ORDERED]
    legend_handles.append(Patch(color=CPM_COLOR, label='CPM'))
    axes[2].legend(
        handles=legend_handles,
        loc='upper right',
        frameon=False,
        fontsize=7,
        handlelength=1.0,
        handleheight=1.0,
        borderpad=0.1,
        labelspacing=0.1,
    )

    stage_label = 'Post-Harmony' if stage == 'post_harmony' else 'Pre-Harmony'
    axes[0].set_ylabel(f'Replicate mAP ({stage_label})')
    fig.tight_layout()

    out_pdf = FIG_DIR / f'{out_stem}.pdf'
    fig.savefig(out_pdf)
    fig.savefig(out_pdf.with_suffix('.png'), dpi=300)
    plt.show()
    print(f'Saved: {out_pdf}')


# Shared y-axis across both stages so the two figures are directly comparable
shared_ymax = wide[VIEWS_ORDERED].values.max() * 1.05

render_modality_figure('post_harmony', 'fig_bg_exploitability_post_harmony', ymax=shared_ymax)
render_modality_figure('pre_harmony',  'fig_bg_exploitability_pre_harmony',  ymax=shared_ymax)

In [ ]:
VIEWS_TBL = ['C', 'CD', 'S', 'SD']

for stage in ['post_harmony', 'pre_harmony']:
    label = 'Post-Harmony' if stage == 'post_harmony' else 'Pre-Harmony'
    df_stage = wide[wide['embedding'] == stage].set_index(['dataset', 'encoder'])

    lines = [f'## {label}', '']
    lines.append('| dataset | encoder | ' + ' | '.join(VIEWS_TBL) + ' |')
    lines.append('|---' * (2 + len(VIEWS_TBL)) + '|')
    for ds in DATASETS:
        for enc in ENCODERS:
            row = df_stage.loc[(ds, enc)]
            cells = [f'{row[v]:.3f}' for v in VIEWS_TBL]
            lines.append(f'| {ds} | {enc} | ' + ' | '.join(cells) + ' |')
        # CPM row: cp_measure operates on segmented cells only → fill S column
        cpm_val = CPM_VALUES[(ds, stage)]
        cpm_cells = ['—', '—', f'{cpm_val:.3f}', '—']
        lines.append(f'| {ds} | CPM | ' + ' | '.join(cpm_cells) + ' |')

    print('\n'.join(lines))
    print()

## Table 5.1 — Replicate mAP across all four paired views (post-Harmony)

Full per-configuration breakdown including the density-patch probes (CD, SD). Useful as the appendix table that backs the main figure.

In [ ]:
def n_perturbations_for(ds: str) -> int:
    return int(raw[raw['dataset'] == ds]['n_perturbations'].iloc[0])

table = post[['dataset', 'encoder', 'C', 'CD', 'S', 'SD',
              'delta_bg', 'delta_CD_C', 'delta_SD_S']].copy()
table.insert(2, 'n_pert', table['dataset'].map(n_perturbations_for))

# Append per-dataset mean rows
mean_rows = []
for ds in DATASETS:
    block = table[table['dataset'] == ds]
    mean_rows.append({
        'dataset': ds,
        'encoder': '— mean —',
        'n_pert': int(block['n_pert'].iloc[0]),
        'C':        block['C'].mean(),
        'CD':       block['CD'].mean(),
        'S':        block['S'].mean(),
        'SD':       block['SD'].mean(),
        'delta_bg': block['delta_bg'].mean(),
        'delta_CD_C': block['delta_CD_C'].mean(),
        'delta_SD_S': block['delta_SD_S'].mean(),
    })

table_full = pd.concat(
    [table, pd.DataFrame(mean_rows)],
    ignore_index=True,
).sort_values(['dataset', 'encoder']).reset_index(drop=True)

display_cols = {
    'C': 'mAP(C)', 'CD': 'mAP(CD)', 'S': 'mAP(S)', 'SD': 'mAP(SD)',
    'delta_bg':  'Δ_bg = C−S',
    'delta_CD_C': 'Δ_CD−C',
    'delta_SD_S': 'Δ_SD−S',
}
print(table_full.rename(columns=display_cols).round(3).to_string(index=False))

# Persist for downstream use (appendix LaTeX export)
csv_path = FIG_DIR / 'table5_1_all_views_post_harmony.csv'
table_full.to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')

## §5.1 prose draft

The paired-view design lets us compute a quantity that single-score evaluation cannot: the background contribution to replicate mAP for each (encoder, dataset) configuration, defined as $\Delta_{\mathrm{bg}} = \mathrm{mAP}(C) - \mathrm{mAP}(S)$. Because $C$ and $S$ contain the same central cell and differ only in whether surrounding pixels are retained or masked, $\Delta_{\mathrm{bg}}$ isolates the performance attributable to non-cell content in the image.

Figure 5.1 shows $\Delta_{\mathrm{bg}}(\mathrm{mAP})$ across all nine (encoder × dataset) configurations on pre-Harmony embeddings. The dominant axis of variation is dataset, not encoder. On RxRx1 (siRNA perturbations), the dataset-mean $\Delta_{\mathrm{bg}}$ is 0.35; on RxRx3-core (CRISPR knock-outs) it is 0.10; on JUMP (compounds) it is 0.03 — a roughly 13-fold range. The gradient is strict: the dataset minimum on RxRx1 (OpenPhenom, 0.052) exceeds the dataset maximum on JUMP (DINO, 0.049). This separation holds identically on cross-batch perturbation recall (R@10 dataset means: 0.32, 0.16, 0.04; Appendix Fig. SX) and is robust to post-Harmony batch correction (Appendix Table SX).

Within each dataset, encoder choice modulates $\Delta_{\mathrm{bg}}$ substantially — on RxRx1, OpenPhenom shows near-zero $\Delta_{\mathrm{bg}}$ (0.05) while DINO and SubCell exceed 0.48 — but even the largest within-dataset spread does not close the gap between modalities. One configuration (OpenPhenom on JUMP) yields a mildly negative $\Delta_{\mathrm{bg}}$ ($-0.005$), consistent with this encoder's minimal background dependence on compound data.

The modality gradient has a natural biological interpretation: siRNA perturbations induce broad transcriptional changes that alter cell density, morphology of neighbouring cells, and local confluence patterns, all of which are visible in the background pixels and absent after segmentation. Compound perturbations act on narrower pathways, producing effects localised to the treated cell's morphology with less impact on the surrounding field of view. CRISPR knock-outs fall between these extremes. CP-BG-Bench makes this gradient measurable; without paired views, it is invisible to standard evaluation.

## Per-perturbation mAP histogram (cp_measure, LW pipeline)

The aggregated mean mAP for cp_measure on Rxrx3-core is ~0.011 even though ~22% of KOs are individually significant (BH q<0.05). The histogram below shows the per-perturbation mAP distribution for all three datasets to disentangle "broad weak signal" from "small subset of strong hits drowned by silent majority".

Source: `evals/baselines/_data/_dl_pipeline_lw/per_pert_<ds>_<stage>.csv` (LW-ZCA + PCA-50 + Harmony, copairs replicate-mAP).

In [ ]:
LW_DIR = Path('../baselines/_data/_dl_pipeline_lw')
LW_DATASETS = [('Rxrx1', 'RxRx1\n(siRNA, HEPG2)'),
               ('Rxrx3C', 'Rxrx3-core\n(CRISPR-KO, HUVEC)'),
               ('JUMP', 'JUMP-CP\n(compounds, U2OS)')]
STAGE_COLOR = {'pre_harmony': '#888888', 'post_harmony': '#2271b2'}
STAGE_LABEL = {'pre_harmony': 'pre-Harmony', 'post_harmony': 'post-Harmony'}

bins = np.linspace(0, 1, 41)

fig, axes = plt.subplots(1, 3, figsize=(FULLWIDTH_IN, 2.2), sharey=False)
for ax, (ds, label) in zip(axes, LW_DATASETS):
    legend_lines = []
    for stage in ('pre_harmony', 'post_harmony'):
        path = LW_DIR / f'per_pert_{ds}_{stage}.csv'
        if not path.exists():
            legend_lines.append(f'{STAGE_LABEL[stage]}: not ready')
            continue
        df = pd.read_csv(path)
        m = df['mean_average_precision'].clip(0, 1)
        sig_q = df['below_corrected_p'].mean()
        ax.hist(m, bins=bins, color=STAGE_COLOR[stage], alpha=0.55,
                edgecolor='black', linewidth=0.3)
        ax.axvline(m.mean(), color=STAGE_COLOR[stage], linewidth=0.9, linestyle='--')
        # legend_lines.append(f"{STAGE_LABEL[stage]}: mean={m.mean():.3f}, sig_q={sig_q:.2f}")
    ax.text(0.97, 0.97, '\n'.join(legend_lines), transform=ax.transAxes,
            ha='right', va='top', fontsize=6.5,
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.85, pad=1.5))
    ax.set_title(label)
    ax.set_xlim(0, 1)
    ax.set_xlabel('per-perturbation mAP')
    ax.grid(axis='y', alpha=0.3, linewidth=0.4)
axes[0].set_ylabel('# perturbations')

# Figure-level legend below the panels (so it never overlaps any histogram)
# legend_handles = [
#     Patch(color=STAGE_COLOR['pre_harmony'],  alpha=0.55, label='pre-Harmony  (mAP on X_pca)'),
#     Patch(color=STAGE_COLOR['post_harmony'], alpha=0.55, label='post-Harmony (mAP on X_pca_harmony)'),
# ]
# fig.legend(handles=legend_handles, loc='lower center',
#            ncol=2, frameon=False, fontsize=7,
#            bbox_to_anchor=(0.5, -0.02))

# for ax, lbl in zip(axes, 'ABC'):
#     label_panel(ax, lbl)

fig.tight_layout(rect=(0, 0.05, 1, 1))
out_pdf = FIG_DIR / 'fig_cpm_lw_per_pert_mAP_hist.pdf'
fig.savefig(out_pdf)
fig.savefig(out_pdf.with_suffix('.png'), dpi=300)
plt.show()
print(f'Saved: {out_pdf}')